In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json
/kaggle/input/datasets/ngocccchannn/llm-embeddings/full_3769_profiles.json
/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet


In [2]:
import time
import os
import json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import sys

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 
from evaluator import evaluate

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
# 1. Đọc dữ liệu phân chia cold-scenario
tr_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')
te_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet')

# 2. Đọc danh sách khóa lạnh test
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json', 'r') as f:
    cold_items_test = json.load(f)

# 3. Nạp ma trận embedding và tệp profile đầy đủ (chứa đúng danh sách course_id theo đúng thứ tự 3.769 khóa)
raw_embeddings = np.load('/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy')
llm_embeddings = np.load('/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy')

# 4. Tạo ma trận đặc trưng ghép theo công thức Phụ lục A.4
nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
fused_embeddings = nz(nz(raw_embeddings) + nz(llm_embeddings))
print(f"Kích thước ma trận embedding ghép (Fused): {fused_embeddings.shape}")

with open('/kaggle/input/datasets/ngocccchannn/llm-embeddings/full_3769_profiles.json', 'r') as f:
    full_profiles = json.load(f)

# Xây dựng từ điển ánh xạ course_id chính xác dựa theo thứ tự dòng trong ma trận embedding (0 đến 3768)
id_to_index = {}
index_to_id = {}
for idx, profile in enumerate(full_profiles):
    # Lấy course_id từ key trong mỗi bản ghi profile
    # Kiểm tra cấu trúc key của profile mẫu
    cid = profile.get('course_id') #or profile.get('id')
    if cid:
        id_to_index[str(cid)] = idx
        index_to_id[idx] = str(cid)

num_total_courses = raw_embeddings.shape[0]
cand_bool_array = np.zeros(num_total_courses, dtype=bool)

matched_count = 0
for cid in cold_items_test:
    str_cid = str(cid)
    if str_cid in id_to_index:
        idx = id_to_index[str_cid]
        if 0 <= idx < num_total_courses:
            cand_bool_array[idx] = True
            matched_count += 1

cold_cand_tensor = torch.tensor(cand_bool_array, dtype=torch.bool)
print(f"Tổng số khóa trong không gian embedding: {num_total_courses}")
print(f"Số lượng khóa lạnh khớp thành công trong cand_tensor: {matched_count} / {len(cold_items_test)}")

cand_bool_array_b = np.ones(num_total_courses, dtype=bool)
cold_cand_tensor_b = torch.tensor(cand_bool_array_b, dtype=torch.bool)
print(f"Số lượng khóa được phép gợi ý ở tập (b): {cold_cand_tensor_b.sum().item()} / {num_total_courses}")

Kích thước ma trận embedding ghép (Fused): (3769, 384)
Tổng số khóa trong không gian embedding: 3769
Số lượng khóa lạnh khớp thành công trong cand_tensor: 502 / 502
Số lượng khóa được phép gợi ý ở tập (b): 3769 / 3769


In [4]:
# 4. Chuẩn hóa ánh xạ user_id sang integer index cho hàm evaluate()
raw_target_dict = te_df.groupby('user_id')['course_id'].apply(list).to_dict()
unique_users = sorted(raw_target_dict.keys())
user_to_int = {u: i for i, u in enumerate(unique_users)}
int_to_user = {i: u for i, u in enumerate(unique_users)}

target_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in gt if str(cid) in id_to_index] 
    for u, gt in raw_target_dict.items() if u in user_to_int
}

raw_mask_dict = tr_df.groupby('user_id')['course_id'].apply(list).to_dict()
mask_dict = {
    user_to_int[u]: [id_to_index[str(cid)] for cid in hist if str(cid) in id_to_index] 
    for u, hist in raw_mask_dict.items() if u in user_to_int
}

user_history = raw_mask_dict

In [5]:
def build_user_embedding_matrix(embeddings_matrix, user_history_dict, c2i):
    """
    Cơ chế Content-KNN: 
    Biểu diễn học viên (u) = trung bình cộng vector embedding của tất cả 
    các khóa học mà học viên đó đã tương tác trong tập huấn luyện (train).
    """
    user_embs = {}
    for u, hist in user_history_dict.items():
        v_idxs = [c2i[str(cid)] for cid in hist if str(cid) in c2i]
        if len(v_idxs) > 0:
            u_emb = embeddings_matrix[v_idxs].mean(axis=0)
            norm = np.linalg.norm(u_emb)
            if norm > 0:
                u_emb = u_emb / norm # Chuẩn hóa L2 vector học viên
            user_embs[u] = u_emb
        else:
            user_embs[u] = np.zeros(embeddings_matrix.shape[1])
    return user_embs



In [6]:
def evaluate_content_knn_model(embeddings_matrix, cand=None, name="Model"):
    print(f"\n================ ĐANG CHẠY ĐÁNH GIÁ: {name} ================")
    
    # 1. Chuẩn hóa L2 ma trận khóa học ứng viên
    norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    norm_embeddings = embeddings_matrix / norms
    
    # 2. Truyền đúng `norm_embeddings` tương ứng của từng biến thể vào hàm build_user_embedding_matrix
    user_embs = build_user_embedding_matrix(norm_embeddings, user_history, id_to_index)
    
    start_time = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        
    # 3. Hàm tính điểm gợi ý: Độ đo tương đồng Cosine giữa vector học viên và embedding khóa học
    def scores_fn(ub_tensor):
        indices = ub_tensor.tolist() if isinstance(ub_tensor, torch.Tensor) else list(ub_tensor)
        batch_users = [int_to_user[int(idx)] for idx in indices]
        
        batch_vecs = np.array([user_embs.get(u, np.zeros(norm_embeddings.shape[1])) for u in batch_users])
        # Phép nhân ma trận tương đương với độ đo Cosine Similarity
        scores = np.dot(batch_vecs, norm_embeddings.T)
        return torch.tensor(scores, dtype=torch.float32)

    # 4. Gọi hàm evaluate với cand là cand_tensor (Khóa lạnh)
    eval_results = evaluate(
        scores_fn=scores_fn, 
        target=target_dict, 
        mask=mask_dict, 
        ks=(5, 10, 20), 
        cand=cand, 
        bs=1024, 
        tie_seed=0
    )
    end_time = time.time()
    total_time = end_time - start_time
    
    # Thống kê phần cứng thực nghiệm theo yêu cầu
    vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
    latency_ms_per_user = (total_time * 1000) / max(1, len(target_dict))
    
    print(f"--- THỐNG KÊ HIỆU NĂNG ({name}) ---")
    print(f"Thời gian chạy (s/epoch): {total_time:.4f} s")
    print(f"Bộ nhớ VRAM (GB): {vram_gb:.4f} GB")
    print(f"Độ trễ (ms/user): {latency_ms_per_user:.4f} ms/user")
    
    print(f"\n--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---")
    for metric_name, values in eval_results.items():
        print(f"{metric_name}: {values.mean():.4f}")
        
    return eval_results

In [7]:
# Chạy đánh giá cho biến thể ghép trên tập (a) (khóa lạnh)
results_fused_a = evaluate_content_knn_model(fused_embeddings, cand=cold_cand_tensor, name="Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập a)")

# Chạy đánh giá cho biến thể ghép trên tập (b) (toàn bộ item)
results_fused_b = evaluate_content_knn_model(fused_embeddings, cand=cold_cand_tensor_b, name="Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập b)")


================ ĐANG CHẠY ĐÁNH GIÁ: Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập a) ================
--- THỐNG KÊ HIỆU NĂNG (Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập a)) ---
Thời gian chạy (s/epoch): 1.4322 s
Bộ nhớ VRAM (GB): 0.0000 GB
Độ trễ (ms/user): 0.4288 ms/user

--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---
Recall@5: 0.1189
Recall@10: 0.1535
Recall@20: 0.2036
NDCG@5: 0.0754
NDCG@10: 0.0870
NDCG@20: 0.1005
HR@5: 0.1263
HR@10: 0.1665
HR@20: 0.2296

================ ĐANG CHẠY ĐÁNH GIÁ: Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập b) ================
--- THỐNG KÊ HIỆU NĂNG (Content-KNN (Ghép: Văn bản gốc + Hồ sơ LLM - Tập b)) ---
Thời gian chạy (s/epoch): 1.6610 s
Bộ nhớ VRAM (GB): 0.0000 GB
Độ trễ (ms/user): 0.4973 ms/user

--- CHỈ SỐ ĐÁNH GIÁ CHI TIẾT ---
Recall@5: 0.0249
Recall@10: 0.0375
Recall@20: 0.0894
NDCG@5: 0.0157
NDCG@10: 0.0199
NDCG@20: 0.0328
HR@5: 0.0260
HR@10: 0.0410
HR@20: 0.0940


============================